### Ingestion del archivo "language.csv"

In [0]:
dbutils.widgets.text("p_environment", "")
v_environment = dbutils.widgets.get("p_environment")

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

#### Paso 1 - Leer el archivo CSV usando "DataFrameReader" de Spark 

In [0]:
from pyspark.sql.types import StructField, StructType, IntegerType, StringType

In [0]:
languages_schema = StructType(fields=[
    StructField("languageId", IntegerType(), False),
    StructField("languageCode", StringType(), True),
    StructField("languageName", StringType(), True)
])

In [0]:
language_df = spark.read \
    .option("header", True) \
    .schema(languages_schema) \
    .csv(f"{bronze_folder_path}/{v_file_date}/language.csv")

#### Paso 2 - Seleccionar sólo las columnas "requeridas"

In [0]:
from pyspark.sql.functions import col

In [0]:
language_selected_df = language_df.select(col("languageId"), col("languageName"))

#### Paso 3 - Cambiar el nombre de las columnas según lo "requerido"

In [0]:
languages_renamed_df = language_selected_df.withColumnsRenamed({"languageId": "language_id", "languageName": "language_name"})

#### Paso 4 - Agregar la columna "ingestion_date" y "enviroment" al DataFrame

In [0]:
from pyspark.sql.functions import lit

In [0]:
languages_final_df = add_ingestion_date(languages_renamed_df) \
    .withColumn("environment", lit(v_environment)) \
    .withColumn("file_date", lit(v_file_date))

#### Paso 5 - Escribir datos en el DataLake en formato "Delta"

In [0]:
languages_final_df.write.mode("overwrite").format("delta").saveAsTable("movie_silver.languages")

In [0]:
%sql
SELECT * FROM movie_silver.languages;

language_id,language_name,ingestion_date,environment,file_date
24574,English,2026-09-21T16:11:29.262099Z,production,2024-12-16
24575,svenska,2026-09-21T16:11:29.262099Z,production,2024-12-16
24576,Deutsch,2026-09-21T16:11:29.262099Z,production,2024-12-16
24577,unknown,2026-09-21T16:11:29.262099Z,production,2024-12-16
24578,Nihongo,2026-09-21T16:11:29.262099Z,production,2024-12-16
24579,Français,2026-09-21T16:11:29.262099Z,production,2024-12-16
24580,Español,2026-09-21T16:11:29.262099Z,production,2024-12-16
24581,al-?arabiyyah,2026-09-21T16:11:29.262099Z,production,2024-12-16
24582,Latin,2026-09-21T16:11:29.262099Z,production,2024-12-16
24583,Khémôrôphéasa,2026-09-21T16:11:29.262099Z,production,2024-12-16


In [0]:
dbutils.notebook.exit("Success")